In [1]:
!pip install -q transformers accelerate pandas
import torch, pandas as pd
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL = "openbmb/MiniCPM5-2B"
tok = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForCausalLM.from_pretrained(MODEL, torch_dtype=torch.float16, device_map="auto")

def ask(prompt):
    msgs = [{"role": "user", "content": prompt}]
    text = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    ids = tok(text, return_tensors="pt").to(model.device)
    out = model.generate(**ids, max_new_tokens=1024, do_sample=False)
    full = tok.decode(out[0][ids.input_ids.shape[1]:], skip_special_tokens=True)
    return full.split("</think>")[-1].strip()

df = pd.read_csv("words.csv")
df["model_output"] = df["roman_urdu"].apply(
    lambda w: ask(f"Write the Roman Urdu word '{w}' in Urdu script. Reply with only the word."))
df.to_csv("results.csv", index=False)

config.json:   0%|          | 0.00/704 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/94.4k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.89M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/5.43k [00:00<?, ?B/s]

...

model.safetensors.index.json:   0%|          | 0.00/31.4k [00:00<?, ?B/s]

model-00000-of-00001.safetensors: reconstructing file:   0%|          |  0.00B / 5.03GB            

model-00000-of-00001.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/381 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/213 [00:00<?, ?B/s]

In [2]:
import pandas as pd
df = pd.read_csv("scored.csv")
print(df.groupby("correct_letter")[["z_letter_correct","correct"]].mean())
print(df["script"].value_counts())
print(pd.crosstab(df["correct_letter"], df["letters_used"]))


                z_letter_correct  correct
correct_letter                           
ذ                       0.000000      0.0
ز                       0.916667      0.5
ض                       0.000000      0.0
ظ                       0.000000      0.0
script
arabic        44
devanagari     2
mixed          2
Name: count, dtype: int64
letters_used    none   ز
correct_letter          
ذ                  1  11
ز                  1  11
ض                  1  11
ظ                  0  12
